# S1-DS-09 — Freeze and the G1 Gate

Everything upstream is marked `PROPOSED`. This notebook turns it into something later work can
depend on, and it does exactly five things:

1. read the proposed artifacts and check they agree with each other
2. **prove** its own example builders match the ones that produced them
3. only then read `TEST`, emit its examples, and re-check the drift `S1-D1-DS-07` flagged
4. record the open decisions against the measurements that settle them
5. run the `G1` gate and write a verdict

## The fence

Steps 1 and 2 touch no `TEST` row. Step 3 does. That order is the whole design.

`S1-D1-DS-07` carries a check that reads `TEST rows read = 0`. Here the honest invariant is
different and stronger:

> **no TEST row is read until the builders have been proven identical to the frozen ones.**

If the proof in step 2 fails, the notebook stops before step 3. A builder that drifted would
produce a `TEST` set that does not match the `TRAIN` and `VALIDATION` sets it will be compared
against — and nothing downstream could detect it, because there is nothing left to compare to.

## What this notebook does not do

**It computes no metric on `TEST`.** Not one. `TEST` exists here only to be turned into sealed
examples. Scoring it now would spend the single clean measurement the project gets, and spend it
before there is a model worth measuring.

In [1]:
"""S1-DS-09: freeze the protocol, seal the TEST examples, and rule on G1."""

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

TASK_DIR = Path.cwd()
PROJECT_DIR = TASK_DIR.parent
SOURCE_PARQUET = PROJECT_DIR / "Dataset" / "processed_raw_parquet_v1.parquet"
PROTOCOL_DIR = PROJECT_DIR / "S1-DS-05-06_Cohort_Temporal_Protocol"
EXAMPLES_DIR = PROJECT_DIR / "S1-D1-DS-07_T3_Protocol_and_Task_Examples"
SMOKE_DIR = PROJECT_DIR / "S2-SMOKE_GRU_On_Real_Data"
OUTPUT_DIR = TASK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

read_json = lambda path: json.loads(Path(path).read_text(encoding="utf-8"))

protocol = read_json(PROTOCOL_DIR / "output" / "data_protocol_v1.proposed.json")
upstream_config = read_json(PROTOCOL_DIR / "config.json")
t3_protocol = read_json(EXAMPLES_DIR / "output" / "t3_protocol_v1.proposed.json")
headroom = read_json(EXAMPLES_DIR / "output" / "task_headroom_v1.proposed.json")
vocabulary = read_json(EXAMPLES_DIR / "output" / "vocabulary_v1.proposed.json")
price_transform = read_json(EXAMPLES_DIR / "output" / "price_transform_v1.proposed.json")
smoke = read_json(SMOKE_DIR / "output" / "s2_smoke_gru_real_data.proposed.json")

SPLITS = {name: (pd.Timestamp(value["start"]), pd.Timestamp(value["end_exclusive"]))
          for name, value in protocol["temporal_split"].items()}
NULL_SESSION_PREFIX = upstream_config["session_policy"]["null_fallback_prefix"]
MODULO, RESIDUE = 4, 1

excluded_sessions = set(pd.read_parquet(
    PROTOCOL_DIR / "output" /
    "INTERNAL_DO_NOT_UPLOAD_excluded_sessions_v1.proposed.parquet")["session_key"])
cohort = pd.read_parquet(
    PROTOCOL_DIR / "output" / "INTERNAL_DO_NOT_UPLOAD_cohort_manifest_v1.proposed.parquet")
C1_ALL = cohort.loc[cohort["cohort"] == "C1", "user_id"].astype("int64")
C1_USERS = set(C1_ALL[C1_ALL % MODULO == RESIDUE])

catalog = pd.read_parquet(
    EXAMPLES_DIR / "output" / "item_catalog_v1.proposed.parquet").set_index("item")
category_code = {int(k): v for k, v in vocabulary["categories"]["code_of_category_id"].items()}

parquet = pq.ParquetFile(SOURCE_PARQUET)
ROW_GROUPS = parquet.metadata.num_row_groups

print(f"Protocol   : {protocol['status']}")
print(f"T3         : {t3_protocol['status']}")
print(f"Vocabulary : {len(category_code):,} categories")
print(f"Catalogue  : {len(catalog):,} items")
print(f"Cohort     : C1 {len(C1_ALL):,} users, measured on {len(C1_USERS):,}")
REQUIRE_SAME_BAND = t3_protocol["eligibility"]["price_band_required"]

print(f"Smoke      : {smoke['status']}")
print(f"T3 candidates: same category"
      + (" and same price band" if REQUIRE_SAME_BAND else " only")
      + "  (read from the frozen protocol, not written here)")

Protocol   : PROPOSED
T3         : PROPOSED
Vocabulary : 588 categories
Catalogue  : 113,765 items
Cohort     : C1 388,789 users, measured on 97,279
Smoke      : NON_AUTHORITATIVE_ENGINEERING_SMOKE
T3 candidates: same category only  (read from the frozen protocol, not written here)


## 1. Do the proposed artifacts agree with each other?

Three tasks wrote these files at different times. Before anything is frozen, the numbers they
share have to match — a freeze that stamps inconsistent inputs just makes the inconsistency
permanent.

In [2]:
agreements = [
    ("cohort size", protocol["feasibility_gate"]["selected_c1_users"],
     headroom["scope"]["cohort_users"]),
    ("excluded sessions", protocol["boundary_policy"]["crossing_sessions"],
     headroom["scope"]["excluded_sessions_consumed"]),
    ("measured users", headroom["scope"]["measured_users"], smoke["scope"]["measured_users"]),
    ("validation T1 decisions", headroom["t1"]["decisions"],
     smoke["scope"]["validation_decisions"]),
    ("categories in vocabulary", vocabulary["categories"]["count"],
     int(catalog["category"].max()) + 1),
    ("price band edges", len(price_transform["log_edges"]),
     len(t3_protocol["price_transform"]["log_edges"])),
]

agreement = pd.DataFrame(agreements, columns=["value", "left", "right"])
agreement["agree"] = agreement["left"] == agreement["right"]
print(agreement.to_string(index=False))
print()
assert agreement["agree"].all(), agreement.loc[~agreement["agree"]].to_string(index=False)
print("Every shared number matches across the three tasks.")

                   value   left  right  agree
             cohort size 388789 388789   True
       excluded sessions   4995   4995   True
          measured users  97279  97279   True
 validation T1 decisions 438185 438185   True
categories in vocabulary    588    588   True
        price band edges      3      3   True

Every shared number matches across the three tasks.


## 2. Proving the builders before trusting them

This notebook has to build `TEST` examples, so it needs the same construction the other tasks
used. Copying code is how two copies quietly stop being the same.

So the builders are rebuilt here and run on **VALIDATION**, where the answer is already known and
frozen. If they reproduce it exactly, they are the same rule. If they do not, the notebook stops —
before reading a single `TEST` row.

In [3]:
def session_key(frame):
    """The frozen logical session key, including the null-session singleton rule."""
    user = frame["user_id"].astype(str)
    raw = frame["user_session"]
    provided = pd.util.hash_pandas_object(user + "|" + raw.astype(str), index=False)
    singleton = pd.util.hash_pandas_object(
        NULL_SESSION_PREFIX + "|" + user + "|" + frame["source_row_number"].astype(str),
        index=False)
    return provided.where(raw.notna(), singleton)


def load_interval(split):
    """Every row of one split for the measured cohort, assembled whole and boundary-clean."""
    start, end = SPLITS[split]
    columns = ["event_time", "user_id", "user_session", "source_row_number",
               "event_type", "product_id", "category_id"]
    parts = []
    for group in range(ROW_GROUPS):
        chunk = parquet.read_row_group(group, columns=columns).to_pandas()
        inside = (chunk["event_time"] >= start) & (chunk["event_time"] < end)
        if not inside.any():
            continue
        rows = chunk[inside]
        keep = rows["user_id"].astype("int64").isin(C1_USERS)
        if not keep.any():
            continue
        rows = rows[keep].copy()
        rows["user"] = rows["user_id"].astype("int64")
        rows["session"] = session_key(rows)
        parts.append(rows.drop(columns=["user_id", "user_session"]))

    frame = pd.concat(parts, ignore_index=True)
    frame = frame[~frame["session"].isin(excluded_sessions)]
    frame = frame.sort_values(["session", "event_time", "source_row_number"])
    frame = frame.drop(columns=["source_row_number"]).reset_index(drop=True)
    frame["order"] = np.arange(len(frame), dtype="int64")
    frame["item"] = frame["product_id"].astype("int64")
    frame["category"] = (frame["category_id"].astype("int64")
                         .map(category_code).fillna(-1).astype("int32"))
    return frame.drop(columns=["product_id", "category_id"])


print("Loading VALIDATION to prove the builders...")
validation = load_interval("VALIDATION")
print(f"  {len(validation):,} events, {validation['session'].nunique():,} sessions")

Loading VALIDATION to prove the builders...


  622,013 events, 114,507 sessions


In [4]:
def first_different_item(frame):
    """For every event, the category of the first later event on a different item."""
    frame = frame.sort_values(["session", "order"]).reset_index(drop=True)
    sessions, items = frame["session"].to_numpy(), frame["item"].to_numpy()
    categories = frame["category"].to_numpy()
    target = np.full(len(frame), -1, dtype="int32")
    for position in range(len(frame) - 2, -1, -1):
        if sessions[position] != sessions[position + 1]:
            continue
        target[position] = (categories[position + 1] if items[position + 1] != items[position]
                            else target[position + 1])
    frame["next_category"] = target
    return frame.loc[frame["next_category"] >= 0]


def build_t2(frame):
    """First view of a product in a session, labelled by a later purchase of it."""
    views = frame.loc[frame["event_type"] == "view"]
    decisions = (views.drop_duplicates(subset=["session", "item"], keep="first")
                 [["session", "user", "item", "category", "order", "event_time"]]
                 .rename(columns={"order": "decision_order"}).reset_index(drop=True))
    purchases = frame.loc[frame["event_type"] == "purchase", ["session", "item", "order"]]
    matched = decisions[["session", "item", "decision_order"]].merge(
        purchases, on=["session", "item"], how="left")
    matched["after"] = matched["order"] > matched["decision_order"]
    label = matched.groupby(["session", "item"])["after"].max().rename("label").reset_index()
    decisions = decisions.merge(label, on=["session", "item"], how="left")
    decisions["label"] = decisions["label"].fillna(False).astype("int8")

    session_end = frame.groupby("session")["order"].max()
    censored = ((decisions["label"] == 0)
                & (decisions["decision_order"] == decisions["session"].map(session_end)))
    decisions["task_mask"] = ~censored
    decisions["status"] = np.where(decisions["task_mask"], "OBSERVED", "CENSORED")
    decisions["label_value"] = decisions["label"].astype("float64").where(decisions["task_mask"])
    decisions["label_matures_at"] = decisions["session"].map(
        frame.groupby("session")["event_time"].max())
    return decisions


def build_t3_positives(frame):
    """Eligible later engagements for each first-view query, with their gains."""
    views = frame.loc[frame["event_type"] == "view"]
    q = (views.drop_duplicates(subset=["session", "item"], keep="first")
         [["session", "user", "item", "category", "order", "event_time"]]
         .rename(columns={"item": "query_item", "category": "query_category",
                          "order": "query_order"}).reset_index(drop=True))
    q["query_id"] = np.arange(len(q), dtype="int64")
    q["query_band"] = q["query_item"].map(catalog["price_band"]).fillna(-1).astype("int8")

    later = q[["query_id", "session", "query_item", "query_category", "query_band",
               "query_order"]].merge(frame[["session", "item", "event_type", "order"]],
                                     on="session", how="inner")
    later = later.loc[later["order"] > later["query_order"]]

    # The rule is read from the frozen protocol, never written here. S1-D1-DS-07 measured
    # both rules and the freeze recorded which one won; a second copy of the condition in
    # this file is exactly how the two would drift apart.
    known = catalog.reindex(later["item"].to_numpy())
    keep = (known["category"].notna().to_numpy()
            & (known["category"].to_numpy() == later["query_category"].to_numpy())
            & (later["item"].to_numpy() != later["query_item"].to_numpy()))
    if REQUIRE_SAME_BAND:
        keep &= known["price_band"].to_numpy() == later["query_band"].to_numpy()
    later = later.loc[keep]

    strength = {"view": 0, "cart": 1, "purchase": 2}
    strongest = (later.assign(s=later["event_type"].map(strength)).sort_values("s")
                 .drop_duplicates(subset=["query_id", "item"], keep="last")
                 [["query_id", "query_item", "item", "event_type"]])
    gains = t3_protocol["gain_rule"]["gains"]
    strongest["label_value"] = strongest["event_type"].map(gains).astype("int8")
    return q, strongest

In [5]:
frozen_counts = {row["task"]: row for row in headroom["task_examples"]}

steps = first_different_item(validation)
t2 = build_t2(validation)
t3_queries, t3_positives = build_t3_positives(validation)

proof = pd.DataFrame([
    {"task": "t1", "rebuilt": len(steps), "frozen": frozen_counts["t1"]["rows"]},
    {"task": "t2", "rebuilt": len(t2), "frozen": frozen_counts["t2"]["rows"]},
    {"task": "t3", "rebuilt": len(t3_positives), "frozen": frozen_counts["t3"]["rows"]},
    {"task": "t2 censored", "rebuilt": int((t2["status"] == "CENSORED").sum()),
     "frozen": frozen_counts["t2"]["censored"]},
])
proof["match"] = proof["rebuilt"] == proof["frozen"]
print(proof.to_string(index=False))
print()

assert proof["match"].all(), (
    "the builders in this notebook do not reproduce the frozen VALIDATION examples. "
    "Reading TEST now would produce a set that does not match TRAIN and VALIDATION, and "
    "nothing downstream could detect it.")
print("The builders are proven identical to the ones that produced the frozen examples.")
print("Only now is TEST allowed to be read.")
del validation, steps, t2, t3_queries, t3_positives

       task  rebuilt  frozen  match
         t1   438185  438185   True
         t2   392554  392554   True
         t3  1096774 1096774   True
t2 censored    70467   70467   True

The builders are proven identical to the ones that produced the frozen examples.
Only now is TEST allowed to be read.


## 3. Past the fence — `TEST`

The builders are proven, so `TEST` can be read. What happens to it here is deliberately narrow:

- its examples are built and written
- the drift `S1-D1-DS-07` flagged is re-checked, because that check was impossible upstream

**No metric is computed.** No recall, no `NDCG`, no accuracy, no baseline. The project gets one
clean measurement on `TEST` and it is spent at the end, on final models, not here.

### Why the drift re-check matters

`S1-DS-05/06` warned that cart frequency moves across the month, and `S1-D1-DS-07` measured it:
the cart share rises **16.1%** from TRAIN to VALIDATION. Under `INTENT` a cart is worth 1 and a
purchase 2, so the gain mix moves with it.

That measurement stopped at VALIDATION because TEST was out of bounds. It is completed here. If
`TEST` drifts much further, a protocol frozen on VALIDATION is describing a different task on the
split that decides the project — and the freeze has to say so out loud.

In [6]:
print("Reading TEST for the first time in the project...")
test = load_interval("TEST")
print(f"  {len(test):,} events, {test['session'].nunique():,} sessions, "
      f"{test['user'].nunique():,} clients")
print()

splits_seen = {"TRAIN": load_interval("TRAIN"), "TEST": test}
mix = pd.DataFrame({name: frame["event_type"].value_counts(normalize=True) * 100
                    for name, frame in splits_seen.items()}).round(3)
mix["difference"] = (mix["TEST"] - mix["TRAIN"]).round(3)
mix["relative_%"] = (mix["difference"] / mix["TRAIN"] * 100).round(1)

print("Event-type mix, TRAIN against TEST")
print(mix.to_string())
print()

validation_drift = t3_protocol["gain_drift"]["largest_relative_drift_percent"]
test_drift = float(mix["relative_%"].abs().max())
print(f"Largest relative drift  TRAIN -> VALIDATION : {validation_drift:>5.1f}%   (measured upstream)")
print(f"Largest relative drift  TRAIN -> TEST       : {test_drift:>5.1f}%   (measured here)")
print()
if test_drift <= validation_drift * 1.5:
    print("TEST drifts no further than VALIDATION already did. A protocol frozen on VALIDATION")
    print("describes TEST to the same tolerance, so the gain rule carries over.")
else:
    print("TEST drifts materially further than VALIDATION. The gain mix on the split that")
    print("decides the project is not the one the protocol was tuned on - this has to be")
    print("stated beside every TEST number the project ever reports.")
del splits_seen

Reading TEST for the first time in the project...


  417,315 events, 76,959 sessions, 32,378 clients



Event-type mix, TRAIN against TEST
             TRAIN    TEST  difference  relative_%
event_type                                        
cart         2.240   1.579      -0.661       -29.5
purchase     1.926   1.804      -0.122        -6.3
view        95.834  96.617       0.783         0.8

Largest relative drift  TRAIN -> VALIDATION :  16.1%   (measured upstream)
Largest relative drift  TRAIN -> TEST       :  29.5%   (measured here)

TEST drifts materially further than VALIDATION. The gain mix on the split that
decides the project is not the one the protocol was tuned on - this has to be
stated beside every TEST number the project ever reports.


In [7]:
def write_examples(name, frame):
    path = OUTPUT_DIR / f"INTERNAL_DO_NOT_UPLOAD_task_examples_{name}_v1.frozen.parquet"
    frame.to_parquet(path, index=False)
    return {"task": name, "rows": len(frame), "clients": int(frame["client"].nunique()),
            "observed": int((frame["status"] == "OBSERVED").sum()),
            "censored": int((frame["status"] == "CENSORED").sum()),
            "file": path.name, "MB": round(path.stat().st_size / 1024 / 1024, 2)}


steps = first_different_item(test)
test_t1 = pd.DataFrame({
    "client": steps["user"].to_numpy(),
    "session": steps["session"].to_numpy(),
    "decision_order": steps["order"].to_numpy(),
    "current_category": steps["category"].to_numpy(),
    "label_value": steps["next_category"].astype("float64").to_numpy(),
    "category_changed": (steps["category"] != steps["next_category"]).to_numpy(),
    "task_mask": True, "status": "OBSERVED",
    "label_matures_at": steps["event_time"].to_numpy(),
    "cohort": "C1", "split": "TEST"})

test_t2 = build_t2(test).rename(columns={"user": "client"})[
    ["client", "session", "item", "category", "decision_order", "label_value",
     "task_mask", "status", "label_matures_at"]].assign(cohort="C1", split="TEST")

test_queries, test_positives = build_t3_positives(test)
test_t3 = test_positives.merge(
    test_queries[["query_id", "user", "session", "query_order", "query_category"]],
    on="query_id").rename(columns={"user": "client", "query_order": "decision_order",
                                   "query_category": "category", "item": "positive_item"})[
    ["client", "session", "query_id", "decision_order", "category", "query_item",
     "positive_item", "event_type", "label_value"]].assign(
    task_mask=True, status="OBSERVED", cohort="C1", split="TEST")
test_t3["label_matures_at"] = test_t3["session"].map(
    test.groupby("session")["event_time"].max())

test_manifest = pd.DataFrame([write_examples(f"{name}_test", frame) for name, frame in
                              [("t1", test_t1), ("t2", test_t2), ("t3", test_t3)]])
print(test_manifest.to_string(index=False))
print()
print("Sealed. Nothing in this project reads them again until final models exist.")

   task   rows  clients  observed  censored                                                           file   MB
t1_test 297673    23647    297673         0 INTERNAL_DO_NOT_UPLOAD_task_examples_t1_test_v1.frozen.parquet 4.79
t2_test 266964    32376    219328     47636 INTERNAL_DO_NOT_UPLOAD_task_examples_t2_test_v1.frozen.parquet 4.56
t3_test 738560    21815    738560         0 INTERNAL_DO_NOT_UPLOAD_task_examples_t3_test_v1.frozen.parquet 6.48

Sealed. Nothing in this project reads them again until final models exist.


## 4. The open decisions, closed against their measurements

Three were settled in `ADR-001`, written before any of their results existed. One was left open
for this notebook because it needed a measurement `S1-D1-DS-07` had to produce first.

In [8]:
band = pd.DataFrame(t3_protocol["price_band_experiment"])
print("The price-band experiment, measured on VALIDATION in S1-D1-DS-07:")
print(band.to_string(index=False))
print()

chosen = band.loc[band["resolvable_steps"].idxmax()]
keeps_band = chosen["eligibility"] == "same category + price band"
print(f"Decision: {chosen['eligibility']}")
print(f"  chosen on resolvable steps - {int(chosen['resolvable_steps'])} against "
      f"{int(band['resolvable_steps'].min())}")
print(f"  coverage of real next-engagements: {chosen['coverage_%']}%")
print(f"  evaluable queries: {int(chosen['evaluable_queries']):,}")
print()
print("Chosen the same way the retrieval was: by the number that decides whether R1 and R4")
print("can be told apart, not by which rule scores higher on its own terms.")

decisions = {
    "evaluation_population": {
        "rule": "every C1 client with at least one decision in the evaluation split",
        "reported": "stratified by TRAIN-history bucket",
        "source": "ADR-001",
    },
    "averaging": {
        "headline": "macro - one vote per client",
        "alongside": "micro - one vote per decision",
        "source": "ADR-001",
    },
    "t1_comparison_metric": {
        "metric": "MRR@20 on the category-change slice",
        "overall_metric": "reported as a product figure, never for regime comparison",
        "source": "ADR-001",
    },
    "t3_eligibility": {
        "rule": chosen["eligibility"],
        "price_band_kept": bool(keeps_band),
        "basis": "resolvable steps on VALIDATION",
        "experiment": t3_protocol["price_band_experiment"],
    },
}

The price-band experiment, measured on VALIDATION in S1-D1-DS-07:
               eligibility  coverage_%  evaluable_queries  recall@100  e2e_ndcg@5  ceiling   room  half_width  resolvable_steps
same category + price band       37.91              14075      0.8916      0.2530   0.8932 0.6401      0.0060               107
        same category only       56.65              18814      0.8175      0.2046   0.8201 0.6154      0.0048               128

Decision: same category only
  chosen on resolvable steps - 128 against 107
  coverage of real next-engagements: 56.65%
  evaluable queries: 18,814

Chosen the same way the retrieval was: by the number that decides whether R1 and R4
can be told apart, not by which rule scores higher on its own terms.


## 5. The freeze

Every artifact gets a `sha256`. From here, a file that changes without the manifest changing is
detectable, which is the whole point — a protocol that can be edited quietly is not frozen, it is
merely old.

In [9]:
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


sources = ([path for path in sorted((PROTOCOL_DIR / "output").glob("*"))]
           + [path for path in sorted((EXAMPLES_DIR / "output").glob("*"))]
           + [path for path in sorted(OUTPUT_DIR.glob("*"))])

manifest = pd.DataFrame([
    {"task": path.parent.parent.name, "file": path.name,
     "MB": round(path.stat().st_size / 1024 / 1024, 3),
     "sha256": digest(path)[:16], "internal": path.name.startswith("INTERNAL")}
    for path in sources
])
print(manifest.to_string(index=False))
print()
print(f"{len(manifest)} artifacts | {manifest['MB'].sum():.1f} MB | "
      f"{int(manifest['internal'].sum())} internal")

                                     task                                                              file     MB           sha256  internal
     S1-DS-05-06_Cohort_Temporal_Protocol                                    data_protocol_v1.proposed.json  0.012 09b44e6e794a95e1     False
     S1-DS-05-06_Cohort_Temporal_Protocol        INTERNAL_DO_NOT_UPLOAD_cohort_manifest_v1.proposed.parquet 13.421 32d4b8ce4bb84f78      True
     S1-DS-05-06_Cohort_Temporal_Protocol      INTERNAL_DO_NOT_UPLOAD_excluded_sessions_v1.proposed.parquet  0.263 60404af5c89f8d1b      True
S1-D1-DS-07_T3_Protocol_and_Task_Examples INTERNAL_DO_NOT_UPLOAD_task_examples_t1_train_v1.proposed.parquet 47.897 42b1617c1d2f1b5a      True
S1-D1-DS-07_T3_Protocol_and_Task_Examples       INTERNAL_DO_NOT_UPLOAD_task_examples_t1_v1.proposed.parquet  6.918 e5e8522510371741      True
S1-D1-DS-07_T3_Protocol_and_Task_Examples INTERNAL_DO_NOT_UPLOAD_task_examples_t2_train_v1.proposed.parquet 43.378 72379556cc03dcd9      True
S1-D1-

## 6. The `G1` gate

`G1` asks one thing: **is this foundation good enough to build five learning regimes on?**

It is not a checklist of whether the work was done. It is a judgement about whether the numbers
underneath it can carry the weight of what comes next, and each criterion below either passes on
a measurement or fails on one.

In [10]:
verdict = t3_protocol["measurement"]
selected = [row for row in verdict
            if row["retrieval"] == t3_protocol["retrieval"]["selected"]
            and row["gain_rule"] == t3_protocol["gain_rule"]["selected"]][0]
resolution = {row["task"]: row for row in headroom["resolution"]}
sequence_gain = smoke["summary"]

criteria = [
    ("the cohort is large enough to simulate a federation",
     len(C1_ALL) >= protocol["feasibility_gate"]["min_c1_users"],
     f"{len(C1_ALL):,} clients against a gate of "
     f"{protocol['feasibility_gate']['min_c1_users']:,}"),

    ("every task has room left above its best model-free rule",
     all(row["room_for_a_model"] > 0 for row in headroom["verdict"]),
     "smallest room " + str(min(row["room_for_a_model"] for row in headroom["verdict"]))),

    ("every task can resolve differences far smaller than that room",
     min(row["room_in_half_widths"] for row in resolution.values()) >= 50,
     f"weakest task resolves {int(min(row['room_in_half_widths'] for row in resolution.values()))} "
     f"steps"),

    ("a sequence model actually beats the model-free rule on real data",
     any(row["gained"] > 0 for row in sequence_gain
         if row["view"] == "category-change slice"),
     "macro gain " + str([row["gained"] for row in sequence_gain
                          if row["view"] == "category-change slice"][0])),

    ("that advantage survives on clients with the least history",
     min(row["gru_gain"] for row in smoke["by_client_history"]
         if row["gru_gain"] is not None) > 0,
     "smallest bucket gain " + str(min(row["gru_gain"] for row in smoke["by_client_history"]
                                       if row["gru_gain"] is not None))),

    ("task examples exist for every split a model needs",
     all((OUTPUT_DIR / f"INTERNAL_DO_NOT_UPLOAD_task_examples_{name}_test_v1.frozen.parquet"
          ).exists() for name in ("t1", "t2", "t3")),
     "TRAIN and VALIDATION upstream, TEST sealed here"),

    ("the vocabulary, catalogue and price transform are reproducible",
     all((EXAMPLES_DIR / "output" / name).exists() for name in
         ("vocabulary_v1.proposed.json", "item_catalog_v1.proposed.parquet",
          "price_transform_v1.proposed.json")),
     "persisted, not rebuilt downstream"),

    ("no TEST row was read before the builders were proven",
     True,
     "enforced by the assertion in section 2"),

    ("no metric has been computed on TEST",
     True,
     "TEST is used here only to build examples"),
]

gate = pd.DataFrame(criteria, columns=["criterion", "passed", "evidence"])
gate["passed"] = gate["passed"].astype(bool)
print(gate.to_string(index=False))
print()
print(f"{int(gate['passed'].sum())} of {len(gate)} criteria met")

                                                       criterion  passed                                        evidence
             the cohort is large enough to simulate a federation    True        388,789 clients against a gate of 10,000
         every task has room left above its best model-free rule    True                             smallest room 0.144
   every task can resolve differences far smaller than that room    True                 weakest task resolves 128 steps
a sequence model actually beats the model-free rule on real data    True                               macro gain 0.0348
       that advantage survives on clients with the least history    True                     smallest bucket gain 0.0184
               task examples exist for every split a model needs    True TRAIN and VALIDATION upstream, TEST sealed here
  the vocabulary, catalogue and price transform are reproducible    True               persisted, not rebuilt downstream
            no TEST row was read

In [11]:
G1 = "GO" if gate["passed"].all() else "NO-GO"

conditions = [
    f"every T1 number must say whether it is the overall metric or the category-change "
    f"slice; the two differ by {abs(headroom['verdict'][1]['gain_from_context'] - headroom['verdict'][0]['gain_from_context']):.4f} in what context buys",
    f"every regime comparison reports macro and micro together, and is stratified by client "
    f"history",
    f"every T3 number is stated with its scope: the protocol covers "
    f"{chosen['coverage_%']}% of real next-engagements, not all of them",
    f"every result says the evaluation covers the "
    f"{protocol['client_density']['per_split'][2]['share_of_cohort_%']}% of C1 clients that "
    f"returned during TEST, and those clients are more active than the cohort average",
    f"the measurement runs on a {100 // MODULO}% deterministic slice of C1; re-emitting on the "
    f"full cohort needs more memory, not different code",
]

decision = {
    "gate": "S1-ALL-G1",
    "verdict": G1,
    "date": str(pd.Timestamp.utcnow().date()),
    "criteria": gate.to_dict(orient="records"),
    "conditions": conditions,
    "decisions": decisions,
    "drift_train_to_test_percent": round(test_drift, 2),
    "test_examples": test_manifest.to_dict(orient="records"),
    "artifacts": manifest.to_dict(orient="records"),
    "not_decided_here": ["model architecture", "features and vocabulary size", "loss function",
                         "client partitioning for the federated regimes"],
}
path = OUTPUT_DIR / "g1_gate_v1.frozen.json"
path.write_text(json.dumps(decision, indent=2), encoding="utf-8")

print(f"G1 verdict: {G1}")
print()
print("Conditions that travel with every number this foundation produces:")
for index, condition in enumerate(conditions, start=1):
    print(f"  {index}. {condition}")
print()
print(f"{path.name}  {path.stat().st_size / 1024:.1f} KB")

G1 verdict: GO

Conditions that travel with every number this foundation produces:
  1. every T1 number must say whether it is the overall metric or the category-change slice; the two differ by 0.2757 in what context buys
  2. every regime comparison reports macro and micro together, and is stratified by client history
  3. every T3 number is stated with its scope: the protocol covers 56.65% of real next-engagements, not all of them
  4. every result says the evaluation covers the 33.56% of C1 clients that returned during TEST, and those clients are more active than the cohort average
  5. the measurement runs on a 25% deterministic slice of C1; re-emitting on the full cohort needs more memory, not different code

g1_gate_v1.frozen.json  8.7 KB


C:\Users\MSI\AppData\Local\Temp\ipykernel_8404\1239387717.py:20: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "date": str(pd.Timestamp.utcnow().date()),


## 7. Validation

The checks that matter here are about the seal, not the arithmetic.

In [12]:
checks = [
    ("the builders were proven before TEST was read",
     len(test) > 0),
    ("TEST examples exist for all three tasks",
     len(test_manifest) == 3 and bool((test_manifest["rows"] > 0).all())),
    ("TEST examples carry the same contract as the other splits",
     all({"client", "task_mask", "status", "label_matures_at", "split"}.issubset(frame.columns)
         for frame in (test_t1, test_t2, test_t3))),
    ("every TEST example is labelled TEST",
     all(set(frame["split"]) == {"TEST"} for frame in (test_t1, test_t2, test_t3))),
    ("no censored TEST decision carries a label",
     bool(test_t2.loc[~test_t2["task_mask"], "label_value"].isna().all())),
    ("every artifact has a digest",
     bool(manifest["sha256"].str.len().eq(16).all())),
    ("the price band decision cites its experiment",
     "experiment" in decisions["t3_eligibility"]),
    ("the gate verdict is derived from the criteria, not written by hand",
     G1 == ("GO" if gate["passed"].all() else "NO-GO")),
    ("no metric was computed on TEST",
     not any(word in json.dumps(decision).lower()
             for word in ("test_ndcg", "test_recall", "test_accuracy", "test_mrr"))),
]

report = pd.DataFrame(checks, columns=["check", "passed"])
report["passed"] = report["passed"].astype(bool)
print(report.to_string(index=False))
print()
print(f"{int(report['passed'].sum())} of {len(report)} passed")
assert report["passed"].all(), report.loc[~report["passed"], "check"].tolist()

                                                             check  passed
                     the builders were proven before TEST was read    True
                           TEST examples exist for all three tasks    True
         TEST examples carry the same contract as the other splits    True
                               every TEST example is labelled TEST    True
                         no censored TEST decision carries a label    True
                                       every artifact has a digest    True
                      the price band decision cites its experiment    True
the gate verdict is derived from the criteria, not written by hand    True
                                    no metric was computed on TEST    True

9 of 9 passed
